# Experiment 8: Feature Drift Detection & Field Audit

**Objective**: Detect emerging fraud patterns and identify unused high-value fields

**Research Question**: How can we detect emerging fraud patterns that our handcrafted features don't capture?

## Summary

| Part | Objective | Result |
|------|-----------|--------|
| **8A** | Drift Detection POC | ✅ No significant drift detected |
| **8C** | Full Field Audit | ✅ Found 6 high-value unused fields |


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

# Load results
with open('../artifacts/drift_monitoring/drift_summary.json') as f:
    drift_results = json.load(f)

with open('../artifacts/drift_monitoring/field_audit_report.json') as f:
    audit_results = json.load(f)

print("Results loaded")


## Part 8A: Drift Detection Results

We monitored 27 features across time windows to detect distribution drift using PSI (Population Stability Index).


In [ ]:
print("="*70)
print("DRIFT DETECTION RESULTS")
print("="*70)
print()
print(f"Reference Window: {drift_results['reference_window']['start'][:10]} to {drift_results['reference_window']['end'][:10]}")
print(f"Current Window:   {drift_results['current_window']['start'][:10]} to {drift_results['current_window']['end'][:10]}")
print()
print(f"Drift Share: {drift_results['drift_share']*100:.1f}%")
print(f"Alert Triggered: {'YES' if drift_results['alert_triggered'] else 'NO'}")
print(f"Drifted Features: {len(drift_results['drifted_features'])}")
print()

# Show top PSI scores
psi_df = pd.DataFrame([
    {'feature': k, 'PSI': v} 
    for k, v in drift_results['feature_psi'].items()
]).sort_values('PSI', ascending=False)

print("Top 10 PSI Scores (threshold = 0.1):")
for _, row in psi_df.head(10).iterrows():
    status = "WARNING" if row['PSI'] >= 0.1 else "OK"
    print(f"  [{status}] {row['feature']}: {row['PSI']:.4f}")


In [ ]:
# Display drift timeseries
display(Image(filename='../artifacts/drift_monitoring/drift_timeseries.png'))


## Part 8C: Full Field Audit

We analyzed ALL 292 fields from `raw_insertions.parquet` to identify fields with fraud correlation that we're NOT using.


In [ ]:
# Display field audit visualization
display(Image(filename='../artifacts/drift_monitoring/field_audit_summary.png'))


In [ ]:
print("="*70)
print("HIGH-VALUE UNUSED FIELDS (correlation >= 0.1)")
print("="*70)
print()
print(f"{'Field':<45} {'Correlation':>12} {'Coverage':>10}")
print("-"*70)

for field in audit_results['high_value_unused']:
    name = field['field']
    corr = field.get('fraud_correlation', 0)
    cov = field.get('coverage', 0) * 100
    print(f"{name:<45} {corr:>12.4f} {cov:>9.1f}%")


## Recommendation: Should We Add New Features (Option B)?

### High-Value Unused Fields Analysis

| Field | Correlation | Coverage | Interpretation |
|-------|-------------|----------|----------------|
| `listing.prices.rent.interval` | **0.374** | 82% | WEEKLY vs MONTHLY rent - strong signal |
| `listing.platforms` | **0.284** | 100% | Which platform - different fraud rates |
| `listing.lister.billing.language` | **0.263** | 98% | Billing language preference |
| `listing.lister.billing.salutation` | **0.128** | 98% | Mr/Ms/etc - demographic signal |
| `bundle.initialPrice` | **0.114** | 98% | Initial bundle price paid |
| `bundle.recurringPrice` | **0.110** | 98% | Recurring bundle price |


In [ ]:
rec = audit_results['recommendation']

print("="*70)
print("FINAL RECOMMENDATION")
print("="*70)
print()
print("+" + "-"*68 + "+")
print("|" + " "*15 + "YES - ADD NEW FEATURES" + " "*31 + "|")
print("+" + "-"*68 + "+")
print()
print("Rationale:")
print("  1. Found 6 unused fields with significant fraud correlation (>0.1)")
print("  2. Top field (rent.interval) has 0.37 correlation - very strong signal")
print("  3. All have >80% coverage - no data quality issues")
print("  4. Low effort (2-3 days) for potentially significant gain")
print()
print("Cost-Benefit:")
print("  Current AUC-PR:      0.64 (+180% vs Seon)")
print("  Expected gain:       +2-5% AUC-PR")
print("  Effort required:     2-3 days")
print("  Risk:                Low (categorical features, well-understood)")
print()
print("Implementation Priority:")
print("  1. listing.prices.rent.interval (0.37 correlation)")
print("  2. listing.platforms (0.28 correlation)")
print("  3. listing.lister.billing.language (0.26 correlation)")
print("  4. bundle.initialPrice (0.11 correlation)")
print("  5. bundle.recurringPrice (0.11 correlation)")


## Experiment 8 Conclusion

| Part | Finding | Action |
|------|---------|--------|
| **8A: Drift Detection** | No significant drift | Continue weekly retraining |
| **8C: Field Audit** | 6 high-value unused fields | **Recommend adding features** |
| **8B: Add Features** | (Future) | Implement if time permits |

### Key Insights

1. **Current features are stable** - no drift-triggered retraining needed
2. **Opportunities exist** - 6 unused fields could improve model by 2-5%
3. **Top opportunity**: `rent.interval` with 0.37 fraud correlation
4. **All priority fields have >80% coverage** - easy to implement

### Next Steps (Option B - If Proceeding)

1. Modify `create_artifacts.py` to add new fields
2. Modify `FeatureProcessor` to use them
3. Retrain model (full 121 windows)
4. Validate improvement

**Estimated Effort**: 2-3 days  
**Expected Outcome**: +2-5% AUC-PR improvement
